In [2]:
import pandas as pd
import numpy as np

final = pd.read_parquet('../../data/02-conferences/auxiliar/periodistas_2021_2023.parquet')

final['periodistas'] = final['periodistas'].str.replace(' y ', ',', regex=False)
final['periodistas'] = final['periodistas'].str.replace(';', ',', regex=False)
final['periodistas'] = final['periodistas'].str.replace('-', '', regex=False)  # Remove dashes
final['periodistas'] = final['periodistas'].str.replace('\n', '|', regex=False)  # Replace newline characters with commas

final['reporters_list'] = final['periodistas'].str.split('|')

# Step 2: Expand the list into separate columns
max_reporters = final['reporters_list'].apply(len).max()  # Find the maximum number of reporters in a single row
reporter_columns = [f"reporter{i+1}" for i in range(max_reporters)]  # Generate column names

final[reporter_columns] = pd.DataFrame(final['reporters_list'].tolist(), index=final.index)

final = final.drop(columns=['reporters_list'])
final

,Texto,month,ddd,y,text_aux,filtered_sentences,periodistas,date,reporter1,reporter2,...,reporter11,reporter12,reporter13,reporter14,reporter15,reporter16,reporter17,reporter18,reporter19,reporter20
0,La Secretaría de la Defensa Nacional informa ...,1,4,2021,La Defensa Nacional Aeropuerto Internacional F...,"Buenos días a todas y a todos. Sí, buenos día...","Héctor Tlatempa, Puntos Suspensivos Radio, Pun...",2021-01-04,"Héctor Tlatempa, Puntos Suspensivos Radio, Pun...","Demian Duarte, Sonora Power, Lobos FM,Polític...",...,None,None,None,None,None,None,None,None,None,None
1,"Buenos días, presidente; buenos días, subsecr...",1,5,2021,Shaila Rosagel Grupo Healy El Imparcial Sonora...,"Shaila Rosagel, corresponsal de Grupo Healy, E...","Shaila Rosagel, Grupo Healy, El Imparcial, de ...",2021-01-05,"Shaila Rosagel, Grupo Healy, El Imparcial, de ...","Sheila,",...,None,None,None,None,None,None,None,None,None,None
2,"Presidente, buenos días. Judith Sánchez Reyes...",1,6,2021,Judith Sánchez Reyes Imagen Golfo Veracruz Ele...,"Judith Sánchez Reyes, corresponsal de Imagen d...","Judith Sánchez Reyes, Imagen del Golfo | Pedro...",2021-01-06,"Judith Sánchez Reyes, Imagen del Golfo","Pedro Villa,Caña, El Universal",...,None,None,None,None,None,None,None,None,None,None
3,"Buenos días, presidente. Shaila Rosagel, corr...",1,8,2021,Shaila Rosagel Grupo Healy El Imparcial Sonora...,"Gracias. Muchas gracias. Presidente, buenos ...","Shaila Rosagel, Grupo Healy, El Imparcial, La ...",2021-01-08,"Shaila Rosagel, Grupo Healy, El Imparcial, La ...","Daniel Marmolejo, La Cuarta República, Voces ...",...,None,None,None,None,None,None,None,None,None,None
4,La Secretaría de la Defensa Nacional informa ...,1,11,2021,La Defensa Nacional Aeropuerto Internacional F...,Si usted en particular va a aplicar a algún t...,"Paul Velázquez, Los Mochis | Francisco Huerta,...",2021-01-11,"Paul Velázquez, Los Mochis","Francisco Huerta, (Inaudible)",...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
650,Gracias. Muy buenos días a todas y a todos. L...,12,19,2023,Muy Liliana Noble Pulso Saludable Time Out Par...,"Muy buenos días a todas y a todos. Gracias, d...","Liliana Noble, Pulso Saludable | Liliana Piña,...",2023-12-19,"Liliana Noble, Pulso Saludable","Liliana Piña, ATiempo.com.mx,PuenteLibre.mx",...,None,None,None,None,None,None,None,None,None,None
651,Continúan los esfuerzos del Gobierno de Méxic...,12,20,2023,Continúan Otis Guerrero Estamos A Tianguis Bie...,"Buenos días, señor presidente. ¿Se va a busca...","Andrés García, Códice 21 | Arturo Páramo, Grup...",2023-12-20,"Andrés García, Códice 21","Arturo Páramo, Grupo Imagen",...,None,None,None,None,None,None,None,None,None,None
652,"La Secretaría de Infraestructura, Comunicacio...",12,21,2023,La Infraestructura Comunicaciones Transportes ...,"Buenos días, señor presidente. Con Agua Salud...","Fernando Olivas, Radio Relax,¿Qué Pasó?, digit...",2023-12-21,"Fernando Olivas, Radio Relax,¿Qué Pasó?, digital",Marisol Cruz Hernández,...,None,None,None,None,None,None,None,None,None,None
653,"Campaña de miedo en Tabasco. Políticos, comun...",12,27,2023,Campaña Tabasco Políticos Villahermosa Tabasco...,"Ernesto Ledesma, de Rompeviento Tv. Jefe de Go...","José Sobrevilla, Noreste| Ernesto Ledesma, Ro...",2023-12-27,"José Sobrevilla, Noreste","Ernesto Ledesma, Rompeviento Tv",...,None,None,None,None,None,None,None,None,None,None


In [9]:
df_dates = final[(final['date']<'2023-04-19') &  (final['date']>'2021-04-19')]

df_dates = df_dates.sample(n=10, random_state=42)['date']
df_dates = df_dates.sort_values().reset_index(drop=True)
df_dates.to_excel('../../data/02-conferences/auxiliar/random_dates_mañanera.xlsx', 
                  index = False)

In [3]:
# Step 1: Dynamically generate the list of reporter columns
value_vars = [col for col in final.columns if col.startswith('reporter')]

# Step 2: Pivot longer (wide to long format)
df_long = pd.melt(final, id_vars=['date'], value_vars=value_vars, 
                  var_name='reporter_type', value_name='reporter')

# Step 3: Filter out rows where 'reporter' is None
df_long = df_long[df_long['reporter'].notna()]

# Step 4: Drop the 'reporter_type' column if not needed
df_long = df_long.drop(columns=['reporter_type'])
df_long = df_long.sort_values(by='date').reset_index(drop=True)
df_long

,date,reporter
0,2021-01-04,"Héctor Tlatempa, Puntos Suspensivos Radio, Pun..."
1,2021-01-04,"Carlos Calzada, Radio Educación"
2,2021-01-04,"Meme Yamel, The México News,Sin Censura"
3,2021-01-04,"Nuri Fernández, La Caracola"
4,2021-01-04,"Demian Duarte, Sonora Power, Lobos FM,Polític..."
...,...,...
2421,2023-12-27,"José Sobrevilla, Noreste"
2422,2023-12-27,"Ernesto Ledesma, Rompeviento Tv"
2423,2023-12-29,"Alberto Ruz, Lhuillier"
2424,2023-12-29,"Juan Hernández, Diario Basta, Grupo Cantón"


In [4]:
# Step 1: Split the text column by commas
df_long['outlet_list'] = df_long['reporter'].str.split(',')

# Step 2: Expand the list into separate columns
max_reporters = df_long['outlet_list'].apply(len).max()  # Find the maximum number of reporters in a single row
reporter_columns = [f"outlet{i+1}" for i in range(max_reporters)]  # Generate column names

df_long[reporter_columns] = pd.DataFrame(df_long['outlet_list'].tolist(), 
                                         index=df_long.index)

df_long = df_long.drop(columns=['outlet_list'])

df_long


,date,reporter,outlet1,outlet2,outlet3,outlet4,outlet5,outlet6,outlet7,outlet8,...,outlet31,outlet32,outlet33,outlet34,outlet35,outlet36,outlet37,outlet38,outlet39,outlet40
0,2021-01-04,"Héctor Tlatempa, Puntos Suspensivos Radio, Pun...",Héctor Tlatempa,Puntos Suspensivos Radio,Puntos Suspensivos Comunicación,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,2021-01-04,"Carlos Calzada, Radio Educación",Carlos Calzada,Radio Educación,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,2021-01-04,"Meme Yamel, The México News,Sin Censura",Meme Yamel,The México News,Sin Censura,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,2021-01-04,"Nuri Fernández, La Caracola",Nuri Fernández,La Caracola,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,2021-01-04,"Demian Duarte, Sonora Power, Lobos FM,Polític...",Demian Duarte,Sonora Power,Lobos FM,Política,RockandRoll Radio,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2421,2023-12-27,"José Sobrevilla, Noreste",José Sobrevilla,Noreste,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2422,2023-12-27,"Ernesto Ledesma, Rompeviento Tv",Ernesto Ledesma,Rompeviento Tv,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2423,2023-12-29,"Alberto Ruz, Lhuillier",Alberto Ruz,Lhuillier,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2424,2023-12-29,"Juan Hernández, Diario Basta, Grupo Cantón",Juan Hernández,Diario Basta,Grupo Cantón,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [5]:
df_long.rename(columns = {'reporter':'name_outlet', 'outlet1':'reporter'}, inplace=True)
df_long

,date,name_outlet,reporter,outlet2,outlet3,outlet4,outlet5,outlet6,outlet7,outlet8,...,outlet31,outlet32,outlet33,outlet34,outlet35,outlet36,outlet37,outlet38,outlet39,outlet40
0,2021-01-04,"Héctor Tlatempa, Puntos Suspensivos Radio, Pun...",Héctor Tlatempa,Puntos Suspensivos Radio,Puntos Suspensivos Comunicación,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,2021-01-04,"Carlos Calzada, Radio Educación",Carlos Calzada,Radio Educación,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,2021-01-04,"Meme Yamel, The México News,Sin Censura",Meme Yamel,The México News,Sin Censura,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,2021-01-04,"Nuri Fernández, La Caracola",Nuri Fernández,La Caracola,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,2021-01-04,"Demian Duarte, Sonora Power, Lobos FM,Polític...",Demian Duarte,Sonora Power,Lobos FM,Política,RockandRoll Radio,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2421,2023-12-27,"José Sobrevilla, Noreste",José Sobrevilla,Noreste,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2422,2023-12-27,"Ernesto Ledesma, Rompeviento Tv",Ernesto Ledesma,Rompeviento Tv,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2423,2023-12-29,"Alberto Ruz, Lhuillier",Alberto Ruz,Lhuillier,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2424,2023-12-29,"Juan Hernández, Diario Basta, Grupo Cantón",Juan Hernández,Diario Basta,Grupo Cantón,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [6]:
# Step 1: Dynamically generate the list of reporter columns
value_vars = [col for col in df_long.columns if col.startswith('outlet')]

# Step 2: Pivot longer (wide to long format)
df_long = pd.melt(df_long, id_vars=['date', 'name_outlet', 'reporter'], 
                  value_vars=value_vars, 
                  var_name='outlet_type', value_name='outlet')

# Step 3: Filter out rows where 'reporter' is None
df_long = df_long[df_long['outlet'].notna()]

# Step 4: Drop the 'reporter_type' column if not needed
df_long = df_long.drop(columns=['outlet_type'])
df_long = df_long.sort_values(by='date').reset_index(drop=True)
df_long

,date,name_outlet,reporter,outlet
0,2021-01-04,"Héctor Tlatempa, Puntos Suspensivos Radio, Pun...",Héctor Tlatempa,Puntos Suspensivos Radio
1,2021-01-04,"Demian Duarte, Sonora Power, Lobos FM,Polític...",Demian Duarte,Lobos FM
2,2021-01-04,"Meme Yamel, The México News,Sin Censura",Meme Yamel,Sin Censura
3,2021-01-04,"Héctor Tlatempa, Puntos Suspensivos Radio, Pun...",Héctor Tlatempa,Puntos Suspensivos Comunicación
4,2021-01-04,"Demian Duarte, Sonora Power, Lobos FM,Polític...",Demian Duarte,RockandRoll Radio
...,...,...,...,...
3584,2023-12-27,"José Sobrevilla, Noreste",José Sobrevilla,Noreste
3585,2023-12-29,"Beatriz Contreras, Gobierno de México",Beatriz Contreras,Gobierno de México
3586,2023-12-29,"Juan Hernández, Diario Basta, Grupo Cantón",Juan Hernández,Diario Basta
3587,2023-12-29,"Juan Hernández, Diario Basta, Grupo Cantón",Juan Hernández,Grupo Cantón


In [27]:
# Function to clean names

import re
import unicodedata

def clean_name(name):
    # Convert to lowercase
    name = name.lower()
    # Remove accents
    name = ''.join(
        c for c in unicodedata.normalize('NFD', name)
        if unicodedata.category(c) != 'Mn'
    )
    # Replace multiple spaces with a single space
    name = re.sub(r'\s+', ' ', name)
    name = re.sub(r'[0-9.]', '', name)
    # Remove non-alphabetic characters (except spaces)
    name = re.sub(r'[^a-z ]', '', name).strip()
    return name

df_long['reporter_aux'] = df_long['reporter'].apply(clean_name)
df_long['outlet_aux'] = df_long['outlet'].apply(clean_name)

In [28]:
df_long['reporter'] = df_long['reporter'].astype(str)
df_long['name_outlet'] = df_long['name_outlet'].astype(str)
df_long['reporter_aux'] = df_long['reporter_aux'].astype(str)
df_long['outlet'] = df_long['outlet'].astype(str)
df_long['outlet_aux'] = df_long['outlet_aux'].astype(str)

In [29]:
df_long.to_parquet('../../data/02-conferences/auxiliar/periodistas_long_2021_2023.parquet')

In [17]:
df_per = df_long.drop_duplicates(['reporter_aux', 'date']).reset_index(drop=True)
df_per_agg_tot = df_per.groupby(['reporter_aux']).size().reset_index(name = 'count')
df_per_agg_tot

,reporter_aux,count
0,denise mendoza,1
1,shaila rosagel,1
2,abel quezada,1
3,adalberto carvajal,1
4,adan augusto lopez hernandez,1
...,...,...
761,ximena barragan,1
762,yesenia peralta,3
763,yusbel carolina,3
764,zeltzin juarez,2


In [ ]:


df_dedup = lt.cluster_rows(df_per, model = "sentence-transformers/multi-qa-mpnet-base-dot-v1",
                           on=['name'], cluster_type = 'agglomerative',
                           cluster_params = {'threshold': 0.8,
                                              "min cluster size": 2})

In [16]:
df_per

,date,name_outlet,reporter,outlet,reporter_aux
0,2021-01-04,"Héctor Tlatempa, Puntos Suspensivos Radio, Pun...",Héctor Tlatempa,Puntos Suspensivos Radio,hector tlatempa
1,2021-01-04,"Demian Duarte, Sonora Power, Lobos FM,Polític...",Demian Duarte,Lobos FM,demian duarte
2,2021-01-04,"Meme Yamel, The México News,Sin Censura",Meme Yamel,Sin Censura,meme yamel
3,2021-01-04,"Carlos Calzada, Radio Educación",Carlos Calzada,Radio Educación,carlos calzada
4,2021-01-04,"Nuri Fernández, La Caracola",Nuri Fernández,La Caracola,nuri fernandez
...,...,...,...,...,...
2179,2023-12-27,"Ernesto Ledesma, Rompeviento Tv",Ernesto Ledesma,Rompeviento Tv,ernesto ledesma
2180,2023-12-27,"José Sobrevilla, Noreste",José Sobrevilla,Noreste,jose sobrevilla
2181,2023-12-29,"Beatriz Contreras, Gobierno de México",Beatriz Contreras,Gobierno de México,beatriz contreras
2182,2023-12-29,"Juan Hernández, Diario Basta, Grupo Cantón",Juan Hernández,Diario Basta,juan hernandez
